# Instagram Reels Data Extraction — Auto-Discovery and Targeted Scraping

## Overview
This notebook implements two data collection strategies for extracting reel-level
engagement data from Instagram, using the Apify cloud scraping platform.

- **Part 1 — Auto-Discovery (Auto-Scroll):** Automatically discovers and extracts
  the latest N reels from a specified Instagram account, simulating the behavior
  of scrolling through a profile's reel feed.
- **Part 2 — Targeted Extraction:** Extracts data for a fixed, pre-specified list
  of 20 reel URLs, representing a modification of the auto-discovery approach for
  cases where specific content needs to be retrieved regardless of recency.

## Methodology
Since cloud notebook environments (e.g. Google Colab) do not support a real
browser session required by browser-automation tools such as Selenium, this
implementation uses **Apify**, a cloud-based web scraping platform, to perform
the actual data collection. The scraping job (the "actor run") is configured and
executed through the Apify Console, and this notebook retrieves the resulting
structured dataset via the Apify API for further processing, filtering, and export.

## Execution Steps
1. Configure and run the scraping actor in Apify Console:
   - Part 1: provide the target account username and a results limit.
   - Part 2: provide the list of specific reel URLs to extract.
2. Once the run completes, retrieve its corresponding Dataset ID.
3. Enter the API token and Dataset ID into the configuration cell for the
   relevant part of this notebook.
4. Execute all cells sequentially to fetch, process, and export the data.

## Configurable Parameters (Part 1)
- `REELS_COUNT`: number of most recent reels to retain
- `SORT_BY`: ranking criterion — `"date"` (most recent) or `"views"` (most viewed)

## Output
Each part exports its results as both CSV and Excel files containing reel
metadata: username, URL, caption, like count, comment count, view count, and
post date.

In [ ]:
!pip install apify-client pandas openpyxl -q
print("✅ Libraries installed!")

---
## PART 1 — Auto-Scrolling Reels (latest N from one account)

Run the actor manually for one target username (see workflow above), then paste its Dataset ID below.


In [ ]:
# ========================= PART 1 CONFIGURATION =========================
APIFY_TOKEN = "apify_api_L98R1bWCN94ld1WiqlEyJyPANzYgIP0T8vYl"     # <-- paste your token from console.apify.com
DATASET_ID_PART1 = "WHdGj4mbhaIDdfRO4" # <-- paste Dataset ID from the run's Storage tab
REELS_COUNT = 5                          # how many LATEST reels to keep (5, 10, 20 ...)
SORT_BY = "date"                          # "date" = most recent first, "views" = most viewed first
# ==========================================================================

print(f"Reels to keep : {REELS_COUNT}")
print(f"Sort by       : {SORT_BY}")

In [ ]:
from apify_client import ApifyClient
import pandas as pd
from IPython.display import display

client = ApifyClient(APIFY_TOKEN)
dataset_p1 = client.dataset(DATASET_ID_PART1)
items_p1 = dataset_p1.list_items().items
raw_df_p1 = pd.DataFrame(items_p1)

print(f" Total rows in dataset: {len(raw_df_p1)}")
print(f" Columns: {raw_df_p1.columns.tolist()}\n")

In [ ]:
# ---------- EXTRACT REEL DATA (Part 1) ----------
rows_p1 = []
for idx, row in raw_df_p1.iterrows():
    rows_p1.append({
        'Username': row.get('ownerUsername', ''),
        'Reel_URL': row.get('url', row.get('postUrl', '')),
        'Caption': str(row.get('caption', ''))[:300],
        'Likes_Count': row.get('likesCount', 0),
        'Comments_Count': row.get('commentsCount', 0),
        'Views_Count': row.get('videoPlayCount', row.get('videoViewCount', 0)),
        'Date_Posted': row.get('timestamp', ''),
    })

df_part1 = pd.DataFrame(rows_p1)

# ---------- SORT & KEEP TOP N ----------
if SORT_BY == "views":
    df_part1 = df_part1.sort_values('Views_Count', ascending=False)
else:
    df_part1 = df_part1.sort_values('Date_Posted', ascending=False)

df_part1 = df_part1.head(REELS_COUNT).reset_index(drop=True)
df_part1.insert(0, 'Rank', range(1, len(df_part1) + 1))

print(f"📊 Top {len(df_part1)} reels (sorted by {SORT_BY}):")
display(df_part1)

# If Views_Count or Date_Posted look empty/all-zero above, check the
# "Columns:" list printed in the previous cell — the actor may use
# slightly different field names; tell Claude what you see and it'll adjust.

In [ ]:
# ---------- SAVE & DOWNLOAD (Part 1) ----------
csv_name_p1 = f"part1_autoscroll_top{REELS_COUNT}_reels.csv"
xlsx_name_p1 = f"part1_autoscroll_top{REELS_COUNT}_reels.xlsx"

df_part1.to_csv(csv_name_p1, index=False, encoding='utf-8')
df_part1.to_excel(xlsx_name_p1, index=False)

print(f"💾 Saved: {csv_name_p1}")
print(f"💾 Saved: {xlsx_name_p1}")

from google.colab import files
files.download(csv_name_p1)
files.download(xlsx_name_p1)

print("\n✅ Part 1 (Auto-Scroll) complete!")

---
## PART 2 — Hardcoded Reels (modification of Part 1)

Run the actor manually with your 20 specific reel URLs as `directUrls` (see workflow above), then paste that run's Dataset ID below. This will be a **different** Dataset ID from Part 1, since it's a separate run.


In [ ]:
# ========================= PART 2 CONFIGURATION =========================
APIFY_TOKEN = "apify_api_L98R1bWCN94ld1WiqlEyJyPANzYgIP0T8vYl"     # <-- same token as Part 1
DATASET_ID_PART2 = "r6VMON7bWM1kCesfd" # <-- paste Dataset ID from the HARDCODED run's Storage tab
# ==========================================================================

print("Config loaded for Part 2.")

In [ ]:
from apify_client import ApifyClient
import pandas as pd
from IPython.display import display

client = ApifyClient(APIFY_TOKEN)
dataset_p2 = client.dataset(DATASET_ID_PART2)
items_p2 = dataset_p2.list_items().items
raw_df_p2 = pd.DataFrame(items_p2)

print(f" Total rows in dataset: {len(raw_df_p2)}")
print(f" Columns: {raw_df_p2.columns.tolist()}\n")

In [ ]:
# ---------- EXTRACT REEL DATA (Part 2) ----------
rows_p2 = []
for idx, row in raw_df_p2.iterrows():
    rows_p2.append({
        'Username': row.get('ownerUsername', ''),
        'Reel_URL': row.get('url', row.get('postUrl', '')),
        'Caption': str(row.get('caption', ''))[:300],
        'Likes_Count': row.get('likesCount', 0),
        'Comments_Count': row.get('commentsCount', 0),
        'Views_Count': row.get('videoPlayCount', row.get('videoViewCount', 0)),
        'Date_Posted': row.get('timestamp', ''),
    })

df_part2 = pd.DataFrame(rows_p2)
df_part2.insert(0, 'Rank', range(1, len(df_part2) + 1))

print(f"📊 Data for {len(df_part2)} hardcoded reels:")
display(df_part2)

# Note: row order may not exactly match the order you pasted the 20 URLs in,
# since the actor can process them in parallel.

In [ ]:
# ---------- SAVE & DOWNLOAD (Part 2) ----------
csv_name_p2 = "part2_hardcoded_20_reels.csv"
xlsx_name_p2 = "part2_hardcoded_20_reels.xlsx"

df_part2.to_csv(csv_name_p2, index=False, encoding='utf-8')
df_part2.to_excel(xlsx_name_p2, index=False)

print(f"💾 Saved: {csv_name_p2}")
print(f"💾 Saved: {xlsx_name_p2}")

from google.colab import files
files.download(csv_name_p2)
files.download(xlsx_name_p2)

print("\n✅ Part 2 (Hardcoded) complete! Both CSVs are now downloaded.")